In [ ]:
import poolparty as pp


Party(outputs=[])

In [2]:
# Basic name_seqs usage with prefix
pp.init()
pool = pp.from_seqs(['AAA', 'CCC', 'GGG', 'TTT'], mode='sequential')
pool = pool.name_seqs(prefix='seq_')
pool.print_library()

pool[0]: seq_length=3, num_states=4
state  name  seq
    0  seq_0  AAA
    1  seq_1  CCC
    2  seq_2  GGG
    3  seq_3  TTT



Pool(id=0, name='pool[0]', op='op[0]:from_seqs', num_states=4)

In [4]:
# Names compose through stacked pools
pp.init()
wt_pool = pp.from_seq('AAACCCGGG').name_seqs(prefix='wt_')
mut_pool = pp.from_seqs(['AAACCCGGT', 'AAACCCTTT'], mode='sequential').name_seqs(prefix='mut_')
combined = pp.stack([wt_pool, mut_pool])
combined.print_library()

pool[2]: seq_length=9, num_states=3
state  name  seq
    0  wt_0  AAACCCGGG
    1  mut_0  AAACCCGGT
    2  mut_1  AAACCCTTT



Pool(id=2, name='pool[2]', op='op[2]:stack', num_states=3)

In [5]:
# RepeatOp adds suffixes: mut_0.v0, mut_0.v1, etc.
pp.init()
template = pp.from_seq('<r>AAACCCGGG</r>')
mutagenized = template.mutagenize('r', mutation_rate=0.2, mode='hybrid', num_hybrid_states=3).name_seqs(prefix='mut_')
repeated = mutagenized.repeat_states(2).name_seqs(prefix='v')
repeated.print_library()

pool[4]: seq_length=None, num_states=6
state  name  seq
    0  mut_0.v0  <r>AAACCCGGG</r>
    1  mut_0.v0  <r>AAtCCCGcG</r>
    2  mut_0.v0  <r>AAACCCGGG</r>
    3  mut_0.v1  <r>AAACCCGGG</r>
    4  mut_0.v1  <r>AAtCCCGcG</r>
    5  mut_0.v1  <r>AAACCCGGG</r>



Pool(id=4, name='pool[4]', op='op[4]:repeat', num_states=6)

In [ ]:
# clear_parent_names=True resets the naming chain
template = pp.from_seq('<r>AAACCCGGG</r>')
mutagenized = template.mutagenize('r', mutation_rate=0.2, mode='hybrid', num_hybrid_states=3).name_seqs(prefix='mut_')
repeated = mutagenized.repeat_states(2).name_seqs(prefix='v', clear_parent_names=True)
repeated.print_library()

In [ ]:
# Complex example: stacked pools with naming
template = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAG</cre>ATTACGG<bc/>AGATCGGA')

wt_pool = template.repeat_states(3).name_seqs(prefix='wt_')

mutagenized_pool = (
    template
    .mutagenize('cre', mutation_rate=0.25, mode='hybrid', num_hybrid_states=3)
    .name_seqs(prefix='mut_')
)

deletion_pool = (
    template
    .deletion_scan('cre', deletion_length=3, positions=slice(None, None, 3), mode='sequential')
    .name_seqs(prefix='del_')
)

# Stack and add barcodes
final_pool = (
    pp.stack([wt_pool, mutagenized_pool, deletion_pool])
    .insert_kmers('bc', length=5, case='lower')
    .clear_nonmolecular_chars()
)

final_pool.print_library()

In [ ]:
# Multi-output operations: breakpoint_scan with naming
template = pp.from_seq('AAACCCGGGTTT')
left, right = pp.breakpoint_scan(template, num_breakpoints=1, mode='sequential')
left = left.name_seqs(prefix='L_').named('left_pool')
right = right.name_seqs(prefix='R_').named('right_pool')

df = left.generate_library(num_cycles=1)
print('Left pool names:', list(df['name']))